## to do:

QoL
- plot nodes at any level of transform, see all_nodes
- automatically perform max. level transform
- plot single node by path
- time [s] instead of coefficient number
- perform nodes2data on inverse
- fix plots

last meeting:

next meeting after 21st march

reconstruction
- inverse transform: reconstruct with nodes of interest (harmonics, subharmonics)
- inverse packet transform: how to reconstruct?

visualisation
- graph neural network (python package) 

read thesis
- read thesis, read about processing wavelet packets
- replicate methodology

analyse data
- kurtosis for each packet
- analyse data

general
- tidy up code, move outside of notebook
- add validation data to report
- upload research papers
- feature extraction and classification is the main goal
- learn about filters
- fix max level check

summer
- update cv
- write up project proposal (1 page)
- split signal into details, machine learning (classification)
- combine wavelet packet with graph neural network (GNN) to classify and extract information (do research)

done:
- check padding
- implement other wavelets
- more padding
- testing with: sine waves 2:1:10 MHz + narrow window + noise | harmonics [0.5, 1, 2, 3] MHz f0 with and without bandpass
- calculate energy and frequency ranges for each node


In [ ]:
# suppresses all testing, does not suppress function definitions
# set to true and run all to define functions, then set to false
# generated sound wave is not suppressed
suppress_all = 0

# 1 Sine wave simulation
This code creates a sound wave using sound_wave().
The sound wave is made up of randomly selected sin waves over a given range of frequencies given by "bandwidth".
The frequencies making up the signal is tracked in the list "frequencies".

The data is padded to the next power of 2 for transforms

Used in each cell titled 'running...'

In [ ]:
import numpy as np
from scipy import signal

def sin(A, frequency, phi, t):
    omega = 2*np.pi*frequency
    sin_wave = A*np.sin(omega*t-phi)
    return sin_wave

def sound_wave(A, frequencies, duration, sampling_rate,  delay = 0, end = 0):
    ''' 
    Returns a randomly generated sound wave made up of "n" 
    sine waves. The frequencies present in the wave is tracked 
    with the variable "frequencies" This will be plotted in a 
    histogram. Creates a single sine wave by default.

    Parameters
    ----------
    A_max: float
        the maximum amplitude of each sin wave.
    bandwidth: float
        the maximum frequency of each sin wave.
    duration: float
        the time duration of the signal.
    sampling_rate: float
        sampling_rate of the sensor in hz
    n: int
        number of sine waves to be added to the signal
    band_range: float
        the frequency is selected from the last "band_range"
        (e.g. 1/4) of the selected bandwidth
    delay: float
        a time delay for the signal.
    end: float
        time that the signal ends at.
    rand: boolean
        choose whether to randomise the frequency, phase, 
        and amplitude of the sin wives. Set to False to 
        suppress randomness.
    fundamental_frequency: float
        if this is passed, n harmonics will be added together 
        to form the wave
    '''

    t = np.arange(0,duration,1/sampling_rate)
    wave = np.zeros(int(len(t)))

    # initialise parameters
    phi = 0

    for frequency in frequencies:
        # generate sine wave
        sin_wave = sin(A,frequency,phi,t)
        # add sine wave to total wave
        wave = wave + sin_wave     
        # track the frequency of the sine wave added

    # add delay and end
    delay_index = find_delay_index(t,delay)
    end_index = find_end_index(t,end)

    wave[:delay_index] = 0
    wave[end_index:] = 0
    
    return wave, t

def noise(standard_deviation, duration, sampling_rate):
    t = np.arange(0,duration,1/sampling_rate)
    noise = np.random.normal(0,standard_deviation,len(t))
    return noise

def find_delay_index(t,delay):
    'finds the index of the start of the delayed signal'
    for delay_index in range(len(t)):
        if t[delay_index] > delay:
            return delay_index

def find_end_index(t,end):
    'finds the index of the end of the cut off signal'
    end_index = len(t) - 1
    for _ in range(len(t)):
        if t[end_index] < end:
            return end_index
        end_index -= 1

def window(data,mode = 'hanning'):
    if mode == 'hanning':
        data *= np.hanning(len(data))
    elif mode == 'gaussian':
        data *= signal.windows.gaussian(len(data),1)
    return data*np.hanning(len(data))

def sci_form(number):
    'returns a string of the scientific form of a number'
    e_power = 0
    while number >= 10:
        number /= 10
        e_power += 1
    if e_power < 3:
        return f'{number}'
    return f'{number}e{e_power}'

### 1.1 Noisy Sin Wave

testing with a sin wave which exists between 'delay' and 'end' seconds.
data_pad is used as the input signal for the rest of the code.

In [ ]:
import matplotlib.pyplot as plt

# set parameters

amp_sin_waves = 10
sampling_rate = 50e6

frequencies = [2e6,10e6] # change frequency content. = frequency of non-randomised wave
min_frequency = min(frequencies)

time_period = 1/min_frequency
signal_duration = 10*time_period # full signal duration, signal exists between delay and end
delay = 2*time_period
end = 8*time_period

gaussian_noise_std = 1 # set the standard devaition of the noise

# generate data
nyquist_frequency = sampling_rate/2

data, t_plot= sound_wave(
    amp_sin_waves, frequencies, signal_duration, 
    sampling_rate, delay, end
    )

# add noise
gaussian_noise = noise(gaussian_noise_std, signal_duration, sampling_rate)
data += gaussian_noise

cutoff_frequency = 2e6
filter_order = 1
w = cutoff_frequency/nyquist_frequency
sos = signal.butter(filter_order,w,'low',fs = sampling_rate,output = 'sos')
filtered_data = signal.sosfilt(sos,data)

fig, ax = plt.subplots(3,1)

ax[0].plot(t_plot,data)
ax[0].set_xlabel('time [s]')
ax[0].set_ylabel('amplitude [m]')
ax[0].set_title('Data')
ax[0].grid(True)

ax[1].plot(t_plot,filtered_data)
ax[1].set_xlabel('time [s]')
ax[1].set_ylabel('amplitude [m]')
ax[1].set_title('Filtered Data')
ax[1].grid(True)

ax[2].hist(frequencies)
ax[2].set_xlabel('frequency [Hz]')
ax[2].set_ylabel('frequency of frequencies')
ax[2].set_title('Frequency content of data')
ax[2].grid(True)

fig.tight_layout()

plt.show()

# for future plots
frequencies_title = []
for frequency in frequencies:
    f = sci_form(frequency)
    frequencies_title.append(f'{f}')

title = f"- signal frequency = {frequencies_title} Hz"


# 2. Packet Transform
A packet transform is defined using nodes. 

Symmetric padding has been implemented.

This was extended to the daubechies dwt as well.

### 2.1 Node class

Creating the node class to generate and store data

Padding implemented:
- symmetric
- zero
- constant
- reflect (same as symmetric)

In [ ]:
import pywt
class Transform():
    '''
    Created for each transform. Stores all of the relevant information about
    the transform

    Parameters
    ----------
    data: array
        the sound wave to be transformed
    wavelet: string
        the chosen wavelet for the transform
    mode: string
        the chosen type of padding
    '''
    def __init__(self, data, wavelet, mode):
        self.wavelet = wavelet
        self.mode = mode
        self.data = data
        self.base_node = BaseNode(data)
        self.levels = {0:[self.base_node]}

    def get_level(self,level):
        'returns a list of nodes for the chosen level'
        return self.levels[level]

    def add_node(self,node):
        'appends a node to node list'
        # initialise the level if necessary
        if len(self.levels) == node.level:
            self.add_level()
        self.levels[node.level].append(node)

    def add_level(self):
        'increments the number of levels by 1'
        self.levels[len(self.levels)] = []

    def create_node(self, parent, a_d):
        'creates a node object, adds it to list of nodes'
        # initialise node object
        node = Node(parent,a_d,self)
        self.add_node(node)

    def calc_frequencies(self,sampling_rate):
        'set the sampling rate to give information about nyquist frequency'
        self.sampling_rate = sampling_rate
        for level in self.levels:
            for node in self.levels[level]:
                node.frequency_range = node.freq_calc(self.sampling_rate)
    
    def show_all_energy(self):
        'prints the energy for each node'
        for level in self.levels:
            print(f'level {level} nodes:')
            for node in self.levels[level]:
                try:
                    lf = sci_form(node.frequency_range[0])
                    uf = sci_form(node.frequency_range[1])
                except AttributeError:
                    lf = sci_form(node.norm_frequency_range[0])
                    uf = sci_form(node.norm_frequency_range[1])
                print(f'node: {node.path} [{lf}-{uf} Hz]\nenergy: {node.energy}')
                print()

    def dwt_results(self):
        'extracts the nodes containing dwt results from a dwt or dwpt transform object'
        results = []
        # append all of the relevant detail nodes
        for level in range(1,len(self.levels)):
            nodes = self.get_level(level)
            detail_node = nodes[1]
            results.insert(0,detail_node)
        # append final approx node
        approx_node = nodes[0]
        results.insert(0,approx_node)
        return results
    
class Node():
    def __init__(self, parent, a_d, transform_object):
        '''
        Stores the information for each set of coefficients in the transform
        Each node has an assigned parent and path e.g. node aaa is the third 
        level approximation coefficients with parent node aa.

        Parameters
        ----------
        parent: object
            the the node's parent node i.e. the node that was filtered to 
            obtain the data in this one.
        a_d: string
            whether a high pass or low pass filter was used to obtain the 
            data for this node. 'a' for low pass, 'd' for high pass
        level: int
            the level of transform the node is in
        transform_object: Transform()
            an object representing the transform the node is part of
        '''
        # set parent, path, and transform object
        self.transform_object = transform_object
        self.parent = parent
        self.path = f'{parent.path}{a_d}' # eg 'aaa' for 3rd level approx
        # calculate coefficients
        filter_bank = pywt_filter_bank(self.transform_object.wavelet)
        if a_d == 'a':
            filter_coeffs = filter_bank[0]
        else:
            filter_coeffs = filter_bank[1]
        self.data = self.filter(filter_coeffs, self.transform_object.mode)
        self.a_d = a_d
        self.level = parent.level+1
        self.energy = self.energy_calc()
        self.norm_frequency_range = self.norm_freq_calc()

    def filter(self, filter_coeffs, mode):
        'performs the convolution between an adapted signal and the filter'
        parent_data = self.parent.data.copy()
        
        length = len(filter_coeffs) - 2
        l2 = length
        if len(parent_data) % 2:
                l2 += 1
        # signal extension
        data = pywt.pad(parent_data,(length,l2),mode)
        # perform convolution
        data = np.convolve(data, filter_coeffs, 'valid')
        # downsample
        data = data[::2]
        return data
    
    def energy_calc(self):
        return np.var(self.data)
    
    def norm_freq_calc(self):
        '''
        calculates an upper and lower bound between 
        0-1 to be multiplied with nyquist frequency
        '''
        parent = self.parent
        
        lower_bound = parent.norm_frequency_range[0]
        upper_bound = parent.norm_frequency_range[1]

        diff = upper_bound - lower_bound
        if self.a_d == 'a':
            upper_bound -= diff/2
        elif self.a_d == 'd':
            lower_bound += diff/2
        return lower_bound,upper_bound
    
    def freq_calc(self,sampling_rate):
        'converts the normalised frequency range to an actual freuqency range'
        nyquist_frequency = sampling_rate/2
        lower_bound = self.norm_frequency_range[0]*nyquist_frequency
        upper_bound = self.norm_frequency_range[1]*nyquist_frequency
        return lower_bound,upper_bound
    
    def reconstruct(self):
        coeffs = pywt.waverec(self.data,self.transform_object.wavelet,self.transform_object.mode)
        return coeffs
    
    def calc_kurtosis(self):
        pass

class BaseNode(Node):
    def __init__(self,data):
        self.parent = None
        self.data = data
        self.path = ''
        self.level = 0
        self.norm_frequency_range = (0,1)
        self.energy = self.energy_calc()
    
def pywt_filter_bank(wavelet = 'haar'):
    'obtains the filter coefficients from the pywt filter bank function'
    w = pywt.Wavelet(wavelet)
    filter_bank = w.filter_bank
    return filter_bank

### 2.2 max level check

Checking if the chosen level of transform is too high

In [ ]:
def max_level_check(data, levels, wavelet):
    'checks if there are too many levels'
    no_sections = int(2**levels)
    filter_length = len(pywt_filter_bank(wavelet)[0])
    error_message = f'''too many levels. length of data is length of data is:\
    {len(data)}, this many levels requires {no_sections} data points
    '''
    # if filter_length > 2:
    #     if len(data)/no_sections < filter_length:
    #         raise ValueError(error_message)
    # elif no_sections > len(data):
    #     raise ValueError(error_message)

### 2.3 Transforms

dwt and dpt inintialise nodes defined above. These return a list of final level node objects and a list of tuples containing all nodes (level,Node())

In [ ]:
def dwpt(data, levels, wavelet, mode = 'symmetric'):
    '''
    daubechies packet transform. Returns a transform object
    which contains all of the results
    '''
    max_level_check(data,levels,wavelet)
    
    if isinstance(data,list):
        data = np.array(data)

    transform_object = Transform(data, wavelet, mode)

    # create approximamtion and detail nodes for each node in the last level
    for level in range(1,levels+1):
        for node in transform_object.get_level(level-1):
            # create nodes
            transform_object.create_node(node,'a')
            transform_object.create_node(node,'d')        

    return transform_object

def dwt(data, levels, wavelet, mode = 'symmetric'):
    'discrete wavelet transform'
    max_level_check(data,levels,wavelet)

    if isinstance(data,list):
        data = np.array(data)
    
    transform_object = Transform(data,wavelet,mode)

    for level in range(1,levels+1):
        # get approx node in the last level
        node = transform_object.get_level(level-1)[0]
        # store in output array for next level
        transform_object.create_node(node,'a')
        transform_object.create_node(node,'d')

    return transform_object

### 2.4 Plotting the dbwp nodes

new plotting scheme for the nodes

In [ ]:
def plot_final_nodes(
    nodes, data, t_plot, frequencies, title, 
    levels, nyquist_frequency, trans_type = 'dwt'
    ):

    # plot data
    fig_1, ax_1 = plt.subplots()
    plot_wave(data, t_plot, ax_1, title)

    # plot transform coefficients all at once
    # plot a histogram of frequency content in data
    fig_2, ax_2 = plt.subplots(2)
    plot_all_final_nodes(nodes, ax_2[0], title)
    plot_frequencies(frequencies, ax_2[1], title) 

    # plot each node separately.
    rows = int(len(nodes)/2)
    if trans_type == 'dwt' and not levels % 2: # if the dwt has an even number of levels,
        rows += 1                            # it needs an additional row for plotting
    fig_3, ax_3 = plt.subplots(rows,2)
    plot_nodes(nodes, ax_3, nyquist_frequency, rows)
    fig_3.set_size_inches(18,2.4*rows)

    fig_1.tight_layout()
    fig_2.tight_layout()
    fig_3.tight_layout()
    
    plt.show()

def plot_nodes(nodes, ax, nyquist_frequency, rows):
    'plots the data from each node separately'

    section_counter = 1
    for row in range(rows):
        for column in range(2):
            # plot coefficients
            try:
                node = nodes[section_counter-1]
            except IndexError:
                break
            if rows == 1:
                axis = ax[column]
            else:
                axis = ax[row, column]
            axis.plot(node.data)
            
            # set labels
            axis.set_xlabel('coefficient number')
            axis.set_ylabel('magnitude')
            axis.grid(True)

            try: # this function was used once for the pywt plot which does not have the same syntax
                lower_frequency = node.norm_frequency_range[0]*nyquist_frequency
                upper_frequency = node.norm_frequency_range[1]*nyquist_frequency

                lf = sci_form(lower_frequency)
                uf = sci_form(upper_frequency)

                title = f'''
                Coefficients, {lf}-{uf} Hz
                [section {section_counter} of {len(nodes)}]
                '''
                axis.set_title(title)
            except AttributeError:
                pass
            section_counter += 1

def plot_all_final_nodes(nodes, ax, title = ''):
    'concatenate the data from all of the nodes and plot'
    output = np.array([])
    for node in nodes:
        output = np.concatenate((output,node.data))
    ax.plot(output)
    ax.set_title(f"Coefficients {title}")
    ax.set_xlabel("Coefficient Number")
    ax.set_ylabel("Coefficient")
    ax.grid(True)

def plot_wave(data, t_plot, ax, title = ''):
    'plots the data'
    ax.plot(t_plot,data)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude [m]")
    ax.set_title(f"Data {title}")
    ax.grid(True)

def plot_frequencies(frequencies, ax, title  =''):
    'creates a histogram of the frequencies present in the synthetic data'
    ax.hist(frequencies)
    ax.set_title(f"Frequency Distribution {title}")
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("Frequency of frequency")
    ax.grid(True)

## 2.5 Testing DB packet

with pywt

the generated graphs have the same values, the pywt graphs are a bit messier because the library handles boundaries differently.

works until db18

### 2.5.1 Test plots

In [ ]:
def plot_pywp_test(wp_object,levels):
    'just for testing'
    nodes = [node.path for node in wp_object.get_level(levels)]

    # concatenante coefficients
    coeffs = []
    for node in nodes:
        node_data = wp_object[node].data
        coeffs = np.concatenate((coeffs,node_data))
    print('length pywt: ', len(coeffs))
    
    # plot coefficients
    plt.plot(coeffs)
    plt.grid(True)
    plt.show()

    return coeffs

def plot_test(nodes):
    'just for testing'

    # concatenate coefficients
    output = np.array([])
    for node in nodes:
        output = np.concatenate((output,node.data))
    print('length output: ', len(output))
    # plot coefficients
    plt.plot(output)
    plt.grid(True)
    plt.show()

    return output

### 2.5.2 Plotting library transform

In [ ]:
# more complete plotting of the wavelet packet object from the library
def plot_pywp(wp_object,levels,nyquist_frequency):
    'more complete plotting of the pywt packet transform'
    nodes = [node.path for node in wp_object.get_level(levels)]
    # plot coefficients
    fig_1, ax_1 = plt.subplots()
    plot_all_pywp(wp_object,ax_1,nodes)

    # plot nodes seperately
    fig_2, ax_2 = plt.subplots(int(len(nodes)/2),2)
    plot_pywp_nodes(wp_object,ax_2,levels,nodes,nyquist_frequency)
    fig_2.set_size_inches(18,2.4*int(len(nodes)/2))

    fig_1.tight_layout()
    fig_2.tight_layout()
    
    plt.show()
    
def plot_pywp_nodes(wp_object, ax, levels, nodes, nyquist_frequency):
    'plot each node seperately'
    rows = int(len(nodes)/2)

    # calculation section frequency
    frequency_diff = nyquist_frequency/2**levels
    lower_frequency = 0
    upper_frequency = lower_frequency + frequency_diff

    section_counter = 1
    for row in range(rows):
        for column in range(2):
            # plot coefficients
            node = nodes[section_counter-1]
            node = wp_object[node]

            if rows == 1:
                axis = ax[column]
            else:
                axis = ax[row, column]
            axis.plot(node.data)
            
            # set labels
            axis.set_xlabel('coefficient number')
            axis.set_ylabel('magnitude')
            axis.grid(True)

            lf = sci_form(lower_frequency)
            uf = sci_form(upper_frequency)
            title = f'''
            Coefficients, {lf}-{uf} Hz
            [section {section_counter} of {len(nodes)}]
            '''
            axis.set_title(title)

            # re-calculate frequencies
            lower_frequency = upper_frequency
            upper_frequency = lower_frequency + frequency_diff

            section_counter += 1

def plot_all_pywp(wp_object,ax,nodes):
    'plot all nodes in one chart'

    coeffs = np.array([])
    for node in nodes:
        node_data = wp_object[node].data
        coeffs = np.concatenate((coeffs,node_data))

    ax.plot(coeffs)
    ax.grid(True)
    ax.set_title('pywt transform')
    ax.set_xlabel('coefficient number')
    ax.set_ylabel('magnitude')

### 2.5.2 My packet result vs pywt result

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    data_test = [1,2,0,1,2,5,0,3,1]
    data_test = 10*data_test

    # transform
    for db_order in range(1,4):            
        wavelet = f'db{db_order}'
        packet_object = pywt.WaveletPacket(data_test, wavelet, 'zero')
        for level in range(1,4): 
            print('db_order: ',db_order,'level: ',level)           
            print('pywt coeffs')
            py_coeffs = plot_pywp_test(packet_object,level)

            print('my coeffs')
            transform_object = dwpt(data_test,level,wavelet,'zero')
            my_coeffs = plot_test(transform_object.get_level(level))

            difference = py_coeffs-my_coeffs
            print('difference between transforms')
            plt.plot(difference)
            plt.show()
            print()
            transform_object.show_all_energy()

## 2.6 Running the Daubechies packet transform

Using the data at the top

Compares with pywt, notes the position of the highest amplited in each case given as the node path e.g.:

- a:        approx 1st level
- d:        detail 1st level
- aa:       approx 2nd level
- aad:      detail of approx 2nd level
- adadad:   6th level node

The transforms are very similar.

data_pad no longer has to be used

### 2.6.1 My packet result

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters
    db_order = 10
    levels = 3

    # transform
    wavelet = f'db{db_order}'
    transform_object = dwpt(data, levels, wavelet)
    final_level_nodes = transform_object.get_level(levels)
    transform_object.calc_frequencies(sampling_rate)
    # find location of maximum amplitude        
    max_amp = 0
    max_node = None

    for node in final_level_nodes:
        for coefficient in node.data:
            if coefficient > max_amp:
                max_amp, max_node = coefficient, node.path

    print(f'''maximum amplitude is {max_amp} at node '{max_node}' ''')

    # create plots
    plot_final_nodes(
        final_level_nodes, data, t_plot, frequencies, title, 
        levels, nyquist_frequency, trans_type = 'dwpt'
            )
    
    transform_object.show_all_energy()


### 2.6.2 pywt result

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters
    db_order = 10
    levels = 3

    # transform
    wavelet = f'db{db_order}'
    packet_object = pywt.WaveletPacket(data, wavelet)
    
    # find location of maximum amplitude        
    max_amp = 0
    max_node = None

    nodes = [node.path for node in packet_object.get_level(levels)]
    for node in nodes:
        node_data = packet_object[node].data
        for coefficient in node_data:
            if coefficient > max_amp:
                max_amp, max_node = coefficient, node

    print(f'''maximum amplitude is {max_amp} at node '{max_node}' ''')

    # plot the coefficients
    plot_pywp(packet_object, levels, nyquist_frequency)

### 2.7 Discrete wavelet transform

### 2.7.1 My results

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters
    db_order = 10
    levels = 2

    wavelet = f'db{db_order}'
    # transform
    transform_object = dwt(data, levels, wavelet)
    final_nodes = transform_object.dwt_results()
    transform_object.calc_frequencies(sampling_rate)
    
    # find location of maximum amplitude        
    max_amp = 0
    max_node = None

    for node in final_nodes:
        for coefficient in node.data:
            if coefficient > max_amp:
                max_amp, max_node = coefficient, node.path

    print(f'''maximum amplitude is {max_amp} at node '{max_node}' ''')

    # create plots
    plot_final_nodes(
        final_nodes, data, t_plot, frequencies, title, 
        levels, nyquist_frequency
        )
    transform_object.show_all_energy()

### 2.7.2 pywt results

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters
    db_order = 10
    levels = 2

    # transform
    coeffs = pywt.wavedec(data, f'db{db_order}', level = levels)
    
    # find location of maximum amplitude        
    max_amp = 0
    max_node = None

    for i in range(len(coeffs)):
        node = coeffs[i]
        for coefficient in node:
            if coefficient > max_amp:
                max_amp, max_node = coefficient, i+1

    print(f'''maximum amplitude is {max_amp} in section {max_node} ''')

    # plot the coefficients       
    rows = levels//2+1
    fig, ax = plt.subplots(levels//2+1,2)
    plot_nodes(coeffs, ax, nyquist_frequency, rows)

    fig.set_size_inches(18,2.4*rows)
    fig.tight_layout()

# 3 Testing

Testing with windowed data and data with harmonics.

### 3.1 Synthetic sine waves

sine waves, 2:1:10 MHz + narrow window + noise

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters for wave data
    amp_sin_waves = 1
    sampling_rate = 30e6

    test_frequencies = [1e6*(i) for i in range(2,3)]

    for frequency in test_frequencies:
        frequencies = [frequency] # change frequency content. = frequency of non-randomised wave

        time_period = 1/min_frequency
        signal_duration = 10*time_period # full signal duration, signal exists between delay and end
        delay = 0#2*time_period
        end = signal_duration#8*time_period

        gaussian_noise_std = 0.01 # set the standard devaition of the noise

        # generate data
        nyquist_frequency = sampling_rate/2

        data, t_plot = sound_wave(
            amp_sin_waves, frequencies, signal_duration, 
            sampling_rate, delay, end
            )
        print('datalength',len(data))
        gaussian_noise = noise(gaussian_noise_std,signal_duration,sampling_rate)
        data += gaussian_noise
        data = window(data)

        levels = 5
        db_order = 10
        wavelet = f'db{db_order}'
        transform_object = dwpt(data,levels,wavelet)
        final_level_nodes = transform_object.get_level(levels)
        transform_object.calc_frequencies(sampling_rate)

        # set title
        f = sci_form(frequency)
        title = f"- signal frequency = {f} Hz"

        # plot everything
        plot_final_nodes(
            final_level_nodes, data, t_plot, frequencies, 
            title, levels, nyquist_frequency
            )
        
        x = plot_test(final_level_nodes)
        
        # transform
        packet_object = pywt.WaveletPacket(data, wavelet,maxlevel = 5)
        print('maxlevel',packet_object.maxlevel)
        pywp_results = plot_pywp_test(packet_object,levels)

        print(x-pywp_results)


        
        
        

### 3.2 Harmonics

Harmonics and sub-harmonics around central frequencies [0, 1, 2, 3]. with and without bandpass filter.

In [ ]:
run = 1

if suppress_all:
    run = 0

if run:
    # set parameters for wave data
    amp_sin_waves = 1
    sampling_rate = 30e6

    nyquist_frequency = sampling_rate/2

    test_frequencies = [0.5e6]#, 1e6, 2e6, 3e6]
    harmonics = [1/4, 1/2, 1, 2, 3]

    for frequency in test_frequencies:
        frequencies = []
        for harmonic in harmonics:
            frequencies.append(frequency*harmonic)

        min_frequency = frequency
        time_period = 1/min_frequency
        signal_duration = 10*time_period # full signal duration, signal exists between delay and end
        delay = 2*time_period
        end = 8*time_period

        gaussian_noise_std = 0.3 # set the standard devaition of the noise

        # generate data
        data, t_plot = sound_wave(
            amp_sin_waves, frequencies, signal_duration, 
            sampling_rate, delay, end
            )

        gaussian_noise = noise(gaussian_noise_std,signal_duration,sampling_rate)
        data += gaussian_noise
        data = window(data)

        cutoff_frequency = 2e6
        filter_order = 1
        w = cutoff_frequency/nyquist_frequency
        sos = signal.butter(filter_order,w,'low',fs = sampling_rate,output = 'sos')
        filtered_data = signal.sosfilt(sos,data)

        levels = 7
        db_order = 9
        wavelet = f'db{db_order}'

        # non filtered transform
        transform_object = dwpt(data,levels,wavelet)
        final_level_nodes = transform_object.get_level(levels)
        transform_object.calc_frequencies(sampling_rate)

        f = sci_form(frequency)
        title = f"- centre frequency = {f} Hz" # set title and plot
        plot_final_nodes(
            final_level_nodes, data, t_plot, frequencies, 
            title, levels, nyquist_frequency
            )
        
        # filtered transform
        transform_object = dwpt(filtered_data,levels,wavelet)
        final_level_nodes = transform_object.get_level(levels)
        # set title and plot
        c = sci_form(cutoff_frequency)
        title = f'''- centre frequency = {f} Hz
        cutoff frequency = {c} Hz
        '''
        
        plot_final_nodes(
            final_level_nodes, filtered_data, t_plot, frequencies, 
            title, levels, nyquist_frequency
            )
        
        transform_object.show_all_energy()

## Appendix
code of the haar transform in java from wikipedia

This code is a very basic implementation of the haar wavelet transform. It doesn't deal with datasets of arbitrary length and doesn't account for the "energy" of the wave by multiplying by $\sqrt{2}$.

public static int[] discreteHaarWaveletTransform(int[] input) {
    // This function assumes that input.length=2^n, n>1
    int[] output = new int[input.length];

    for (int length = input.length / 2; ; length = length / 2) {
        // length is the current length of the working area of the output array.
        // length starts at half of the array size and every iteration is halved until it is 1.
        for (int i = 0; i < length; ++i) {
            int sum = input[i * 2] + input[i * 2 + 1];
            int difference = input[i * 2] - input[i * 2 + 1];
            output[i] = sum;
            output[length + i] = difference;
        }
        if (length == 1) {
            return output;
        }

        //Swap arrays to do next iteration
        System.arraycopy(output, 0, input, 0, length);
    }
}

In [ ]:
# D4-D20 coefficients from wikipedia
# 
# 1	0.6830127	0.47046721	0.32580343	0.22641898	0.15774243	0.11009943	0.07695562	0.05385035	0.03771716
# 1	1.1830127	1.14111692	1.01094572	0.85394354	0.69950381	0.56079128	0.44246725	0.34483430	0.26612218
#     0.3169873	0.650365	0.89220014	1.02432694	1.06226376	1.03114849	0.95548615	0.85534906	0.74557507
#     −0.1830127−0.19093442	−0.03957503	0.19576696	0.44583132	0.66437248	0.82781653	0.92954571	0.97362811
#              −0.12083221	−0.26450717	−0.34265671	−0.31998660	−0.20351382	−0.02238574	0.18836955	0.39763774
#                 0.0498175	0.0436163	−0.04560113	−0.18351806	−0.31683501	−0.40165863	−0.41475176	−0.35333620
#                       0.0465036	0.10970265	0.13788809	0.1008467	6.68194092 × 10−4 −0.13695355	−0.27710988
#                           −0.01498699	−0.00882680	0.03892321	0.11400345	0.18207636	0.21006834	0.18012745
#                                       −0.01779187	−0.04466375	−0.05378245	−0.02456390	0.043452675	0.13160299
#                                     4.71742793 × 10−3	7.83251152 × 10−4	−0.02343994	−0.06235021	−0.09564726	−0.10096657
#                                     6.75606236 × 10−3	0.01774979	0.01977216	3.54892813 × 10−4	−0.04165925
#                                    −1.52353381 × 10−3	6.07514995 × 10−4	0.01236884	0.03162417	0.04696981
#                                        −2.54790472 × 10−3	−6.88771926 × 10−3	−6.67962023 × 10−3	5.10043697 × 10−3
#                                         5.00226853 × 10−4	−5.54004549 × 10−4	−6.05496058 × 10−3	−0.01517900
#                                                             9.55229711 × 10−4	2.61296728 × 10−3	1.97332536 × 10−3
#                                                             −1.66137261 × 10−4	3.25814671 × 10−4	2.81768659 × 10−3
#                                                                                 −3.56329759 × 10−4	−9.69947840 × 10−4
#                                                                                 5.5645514 × 10−5	−1.64709006 × 10−4
#                                                                                                     1.32354367 × 10−4
#                                                                                                     −1.875841 × 10−5